In [ ]:
import numpy as np
from hdbscan import HDBSCAN
from umap import UMAP

def get_cluster_centroids(embeddings_gpu, cluster_labels: np.ndarray):
    unique_labels = np.unique(cluster_labels)
    cluster_centroids = {}
    for label in unique_labels:
        if label != -1:  # Skip noise points
            cluster_embeddings = embeddings_gpu[cluster_labels == label]
            cluster_centroid = np.mean(cluster_embeddings, axis=0)
            cluster_centroids[label] = cluster_centroid
    return cluster_centroids

def get_cluster_stats(cluster_labels: np.ndarray, prefix=""):
    cluster_stats = np.unique(cluster_labels, return_counts=True)
    return {
        f"{prefix}clusters_count": len(cluster_stats[0]),
        f"{prefix}noise_count": int(cluster_stats[1][0])
        if -1 in cluster_stats[0]
        else 0,
    }

In [ ]:
import polars as pl
df = pl.read_parquet("../data/interests_clusters/cm0i27jdj0000aqpa73ghpcxf.snappy")

In [ ]:
# Convert the embeddings to a CuPy array
embeddings = df["interests_embeddings"].to_numpy()
fine_cluster_labels = df["cluster_label"].to_numpy()

# Calculate centroids of fine clusters
fine_cluster_centroids = get_cluster_centroids(
    embeddings, fine_cluster_labels
)

In [ ]:
# Reduce the embeddings dimensions
umap_model = UMAP(
    n_neighbors=15, n_components=100, min_dist=0.1, metric="euclidean", verbose=True,
)
reduced_data = umap_model.fit_transform(list(fine_cluster_centroids.values()))


In [ ]:
from typing import Dict
from sklearn.cluster import AgglomerativeClustering

agglomerator = AgglomerativeClustering(
    n_clusters=100, 
    distance_threshold=None, 
    metric="euclidean", 
    linkage="ward", 
)

merged_cluster_labels = agglomerator.fit_predict(
    list(fine_cluster_centroids.values())
    #reduced_data
)

In [ ]:
from sklearn.metrics import pairwise_distances

distances = pairwise_distances(reduced_data, metric="euclidean")

distances[2124][1573]




In [ ]:
reduced_data[0]

In [ ]:
np.unique(merged_cluster_labels, return_counts=True)


In [ ]:
labels = np.array(
    [merged_cluster_labels[label] if label != -1 else label for label in fine_cluster_labels]
)


In [ ]:
result = df.drop("merged_cluster_label").with_columns(
    merged_cluster_label_v2=pl.Series(labels),
)

In [ ]:
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

d = show(result.to_pandas())
d.open_browser()